In [ ]:
from torch_geometric_temporal.nn.recurrent import TGCN
from torch_geometric_temporal.signal import temporal_signal_split
from torch_geometric.utils.convert import from_networkx
from torch_geometric.utils import negative_sampling
from torch_geometric.nn import LayerNorm
import torch.nn.functional as F
import torch.nn as nn
from torch_geometric.nn import GCNConv, Linear, GAE
from torch_geometric_temporal.nn import GConvGRU
from torch_geometric.data import Data
import torch_geometric.transforms as T
from sklearn.metrics import roc_auc_score, roc_curve, precision_recall_curve, confusion_matrix, f1_score, average_precision_score
from sklearn.impute import KNNImputer
from sklearn.model_selection import train_test_split
import networkx as nx
import random
import torch
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

In [ ]:
# Define the path to the directory containing your Excel files
filteredDataPath = "Data/FilteredData/"
matrixAdjacencyPath = "Data/MatrixAdjacency/"

# Get the list of all Excel files in this directory
filteredDataFiles = [f for f in os.listdir(filteredDataPath) if f.endswith('.xlsx') or f.endswith('.xls')]
matrixAdjacencyFiles = [f for f in os.listdir(matrixAdjacencyPath) if f.endswith('.xlsx') or f.endswith('.xls')]

In [ ]:
all_node_names = set()
for file in matrixAdjacencyFiles:
    df = pd.read_excel(os.path.join(matrixAdjacencyPath, file), index_col=0)
    all_node_names.update(df.columns)

# Convert to a sorted list for consistency
all_node_names = sorted(list(all_node_names))

# Initialize tensors with zeros for all stocks
tensors = {file: torch.zeros((len(all_node_names), 1)) for file in filteredDataFiles}

In [ ]:
# Function to check and impute missing values with KNN
for file in filteredDataFiles:
    print(f"Processing file {file}:")

    # Read the Excel file
    data = pd.read_excel(os.path.join(filteredDataPath, file))
    
    # Drop the date column
    if 'Date' in data.columns:
        data = data.drop('Date', axis=1)

    # Use KNN imputer for missing values
    imputer = KNNImputer(n_neighbors=5)
    data_imputed = pd.DataFrame(imputer.fit_transform(data), columns=data.columns)

    # Update tensors with the imputed values
    for column in data_imputed.columns:
        if column in all_node_names:
            # Calculate the mean of each column
            mean_value = data_imputed[column].mean()

            # Find the index of the column in all_node_names
            node_idx = all_node_names.index(column)

            # Update the tensor for this file
            tensors[file][node_idx, 0] = mean_value

    print("=" * 100)

In [ ]:
def read_and_standardize_matrix(file_path, all_node_names):
    df = pd.read_excel(file_path, index_col=0)

    # Create a DataFrame with zeros
    standardized_df = pd.DataFrame(0, index=all_node_names, columns=all_node_names)

    # Update the DataFrame with existing values from df
    for row in df.index:
        for col in df.columns:
            if row in all_node_names and col in all_node_names:
                standardized_df.loc[row, col] = df.loc[row, col]

    # Set diagonal values to 1
    np.fill_diagonal(standardized_df.values, 1)

    return standardized_df

def read_excel_files(folder_path, all_node_names):
    """
    Read all Excel files from the specified folder, clean up the matrices, 
    and store each in a list of DataFrames.

    Parameters:
    - folder_path (str): Path to the folder containing Excel files.

    Returns:
    - list: A list of DataFrames, each representing an Excel file's content.
    """

    # List all Excel files in the folder
    all_files = [f for f in os.listdir(folder_path) if os.path.isfile(os.path.join(folder_path, f)) and f.endswith('.xlsx')]
    
    # List to store DataFrames
    dfs_list = []
    
    for file in all_files:
        file_path = os.path.join(folder_path, file)
        standardized_df = read_and_standardize_matrix(file_path, all_node_names)
        dfs_list.append(standardized_df)

    return dfs_list

In [ ]:
adj_matrices = read_excel_files(matrixAdjacencyPath, all_node_names)

In [ ]:
# Convert adjacency matrices to PyTorch tensor format
adj_tensors = [torch.tensor(df.values, dtype=torch.float32) for df in adj_matrices]

# Convert the dictionary of tensors to a list, keeping order same as adj_matrices
feature_tensors = [tensors[file] for file in filteredDataFiles]

# Convert feature tensors and adjacency tensors to PyTorch Geometric Data objects
def adj_to_edge_list(adj_tensor):
    return adj_tensor.nonzero(as_tuple=True)

In [ ]:
# Convert feature tensors and adjacency tensors to PyTorch Geometric Data objects
data_list = []
for i in range(len(adj_tensors)):
    edge_index = adj_to_edge_list(adj_tensors[i])
    data = Data(x=feature_tensors[i], edge_index=torch.stack(edge_index))
    data.file_name = matrixAdjacencyFiles[i]
    data_list.append(data)

In [ ]:
# Prepare pairwise labels for link prediction
labels = []
for i in range(len(data_list) - 1):
    next_edge_index = data_list[i+1].edge_index
    present_edge_set = set(tuple(edge) for edge in data_list[i].edge_index.t().tolist())
    next_edge_set = set(tuple(edge) for edge in next_edge_index.t().tolist())
    positive_edges = next_edge_set - present_edge_set
    negative_edges = present_edge_set - next_edge_set
    edge_labels = [1]*len(positive_edges) + [0]*len(negative_edges)
    edges = list(positive_edges) + list(negative_edges)
    labels.append((torch.tensor(edges, dtype=torch.long).t(), torch.tensor(edge_labels, dtype=torch.float32)))

In [ ]:
class TemporalGraphDataset:
    def __init__(self, data_list):
        self.data_list = data_list
        self.snapshot_count = len(data_list)

    def __getitem__(self, index):
        return self.data_list[index]

    def __len__(self):
        return self.snapshot_count

    def get_indices_of_files(self, file_names):
        return [i for i, data in enumerate(self.data_list) if data.file_name in file_names]

In [ ]:
def custom_temporal_signal_split(dataset, specific_train_files, train_ratio=0.8):
    # Indices of specific files for the training set
    specific_train_indices = set(dataset.get_indices_of_files(specific_train_files))

    # Indices for all other files
    other_indices = set(range(len(dataset))) - specific_train_indices

    # Split the other indices based on the train_ratio
    train_size = int(train_ratio * len(other_indices))
    other_train_indices = set(sorted(other_indices)[:train_size])

    # Combine specific train files indices with other train indices
    combined_train_indices = specific_train_indices.union(other_train_indices)

    # The rest are for testing
    test_indices = other_indices - other_train_indices

    # Creating the train and test datasets
    train_dataset = TemporalGraphDataset([dataset[i] for i in combined_train_indices])
    test_dataset = TemporalGraphDataset([dataset[i] for i in test_indices])

    return train_dataset, test_dataset

In [ ]:
# Specific files to be included in the training dataset
specific_train_files = ['rectified_matrix_2016-11-04.xlsx', 'rectified_matrix_2008-10-10.xlsx', 'rectified_matrix_2020-02-28.xlsx']

# Wrap your data_list in the TemporalGraphDataset class
temporal_graph_sequence = TemporalGraphDataset(data_list)

# Split the data using the custom function
train_dataset, test_dataset = custom_temporal_signal_split(temporal_graph_sequence, specific_train_files, train_ratio=0.8)

In [ ]:
from torch_geometric.nn import LayerNorm
class DeepTemporalGraphNetwork(torch.nn.Module):
    def __init__(self, num_features, hidden_dim):
        super(DeepTemporalGraphNetwork, self).__init__()
        
        # Initial embedding layer
        self.embedding = nn.Linear(num_features, hidden_dim)
        
        # TGCN layers
        self.tgcn1 = TGCN(hidden_dim, hidden_dim)
        self.tgcn2 = TGCN(hidden_dim, hidden_dim)
        self.tgcn3 = TGCN(hidden_dim, hidden_dim)
        self.tgcn4 = TGCN(hidden_dim, hidden_dim)
        self.tgcn5 = TGCN(hidden_dim, hidden_dim)
        self.tgcn6 = TGCN(hidden_dim, hidden_dim)
        self.tgcn7 = TGCN(hidden_dim, hidden_dim)
        self.tgcn8 = TGCN(hidden_dim, hidden_dim)
        
        # Decoder for link prediction
        self.decoder = nn.Linear(hidden_dim, hidden_dim)
        # Dropout layer for regularization
        self.dropout = nn.Dropout(0.3)
        
        self.norm1 = LayerNorm(hidden_dim)
        self.norm2 = LayerNorm(hidden_dim)
        self.norm3 = LayerNorm(hidden_dim)
        self.norm4 = LayerNorm(hidden_dim)
        self.norm5 = LayerNorm(hidden_dim)
        self.norm6 = LayerNorm(hidden_dim)
        self.norm7 = LayerNorm(hidden_dim)
        self.norm8 = LayerNorm(hidden_dim)
        
    
    def forward(self, x, edge_index):
        # Embedding layer
        x = self.embedding(x)

        # TGCN layers with LeakyReLU activations, dropout, and layer normalization
        identity = x
        x = self.dropout(F.leaky_relu(self.norm1(self.tgcn1(x, edge_index))))
        x = self.dropout(F.leaky_relu(self.norm2(self.tgcn2(x, edge_index)))) + identity # Residual connection

        identity = x
        x = self.dropout(F.leaky_relu(self.norm3(self.tgcn3(x, edge_index))))
        x = self.dropout(F.leaky_relu(self.norm4(self.tgcn4(x, edge_index)))) + identity # Another residual connection

        identity = x
        x = self.dropout(F.leaky_relu(self.norm5(self.tgcn5(x, edge_index))))
        x = self.dropout(F.leaky_relu(self.norm6(self.tgcn6(x, edge_index)))) + identity # Another residual connection

        identity = x
        x = self.dropout(F.leaky_relu(self.norm7(self.tgcn7(x, edge_index))))
        x = self.dropout(F.leaky_relu(self.norm8(self.tgcn8(x, edge_index)))) + identity # Another residual connection
        
        # Decoder for link prediction
        x = self.decoder(x)
        
        # Compute scores for potential links
        link_scores = torch.matmul(x, x.t())
        
        return link_scores

In [ ]:
node_features = 1
hidden_channels = 128  # Keep the increased hidden channels
model = DeepTemporalGraphNetwork(node_features, hidden_channels)
criterion = torch.nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)  # Keep the updated optimizer

print(model)

In [ ]:
# Efficient negative sampling function
def negative_sampling(edge_index, num_nodes, num_neg_samples):
    """
    Sample negative edges (edges that don't exist in the graph) uniformly at random.
    """
    # Create a set of existing edges
    existing_edges = set(tuple(edge) for edge in edge_index.t().tolist())

    # Generate negative samples
    negative_samples = []
    while len(negative_samples) < num_neg_samples:
        # Sample two random nodes
        source = random.randint(0, num_nodes-1)
        target = random.randint(0, num_nodes-1)

        # Check if they form a positive edge or if they've been sampled before
        if (source, target) not in existing_edges and (target, source) not in existing_edges:
            negative_samples.append([source, target])

    return torch.tensor(negative_samples, dtype=torch.long)

In [ ]:
# Import necessary libraries for plotting
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, precision_recall_curve
import numpy as np

# Define function to plot ROC curve
def plot_roc_curve(fpr, tpr, label=None, fpr_range=(0.0, 0.1)):
    plt.figure()
    plt.plot(fpr, tpr, linewidth=2, label=label)
    plt.xlim(fpr_range)
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('Receiver Operating Characteristic (ROC)')
    if label:
        plt.legend(loc="lower right")
    plt.grid(True)
    file_name = f'roc_curve_{np.random.randint(1e8)}.png'
    plt.savefig(file_name)
    plt.close()
    print(f"ROC curve saved as {file_name}")

# Define function to plot Precision-Recall curve
def plot_precision_recall_curve(precision, recall, label=None, recall_range=(0.0, 0.1)):
    plt.figure()
    plt.plot(recall, precision, linewidth=2, label=label)
    plt.xlim(recall_range)
    plt.ylim([0.0, 1.05])
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.title('Precision-Recall Curve')
    if label:
        plt.legend(loc="lower left")
    plt.grid(True)
    file_name = f'pr_curve_{np.random.randint(1e8)}.png'
    plt.savefig(file_name)
    plt.close()
    print(f"Precision-Recall curve saved as {file_name}")

In [ ]:
# Define evaluation function with plotting
def eval_link_predictor_with_plots(model, data, true_edges, labels):
    model.eval()
    with torch.no_grad():
        embeddings = model(data.x, data.edge_index)
        logits = torch.sum(embeddings[true_edges[:, 0]] * embeddings[true_edges[:, 1]], dim=1)
        probs = torch.sigmoid(logits).cpu().numpy()
        
        if len(np.unique(probs)) == 1:
            print("Warning: Model predictions are constant.")
            return {"AUC-ROC": np.nan, "AUC-PR": np.nan, "F1": np.nan}
        
        fpr, tpr, _ = roc_curve(labels, probs)
        auc_roc = roc_auc_score(labels, probs)
        
        precision, recall, _ = precision_recall_curve(labels, probs)
        auc_pr = average_precision_score(labels, probs)
        
        preds = (probs > 0.5).astype(int)
        f1 = f1_score(labels, preds)
        
        plot_roc_curve(fpr, tpr, label=f"AUC-ROC: {auc_roc:.3f}", fpr_range=(0.0, 0.1))
        plot_precision_recall_curve(precision, recall, label=f"AUC-PR: {auc_pr:.3f}", recall_range=(0.0, 0.1))
        
    return {"AUC-ROC": auc_roc, "AUC-PR": auc_pr, "F1": f1}

In [ ]:
# Define Monte Carlo Experiment Function
def monte_carlo_experiment(data_list, specific_train_files, num_experiments=1):
    auc_rocs = []
    auc_prs = []
    f1_scores = []
    all_results = []

    for _ in range(num_experiments):
        # Split the data using the custom function
        train_dataset, test_dataset = custom_temporal_signal_split(TemporalGraphDataset(data_list), specific_train_files, train_ratio=0.8)
        
        # Split the train_dataset into train and validation sets
        train_indices, val_indices = train_test_split(range(len(train_dataset)), test_size=0.1, random_state=random.randint(0, 10000))
        train_subset = TemporalGraphDataset([train_dataset[i] for i in train_indices])
        val_subset = TemporalGraphDataset([train_dataset[i] for i in val_indices])

        # Initialize the model
        model = DeepTemporalGraphNetwork(node_features, hidden_channels)
        criterion = torch.nn.BCEWithLogitsLoss()
        optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=1e-5)

        best_val_loss = float('inf')
        patience = 10
        patience_counter = 0

        # Training loop with early stopping
        total_loss = 0
        for epoch in range(150):
            model.train()
            for snapshot_idx, snapshot in enumerate(train_subset):
                optimizer.zero_grad()
                embeddings = model(snapshot.x, snapshot.edge_index).squeeze()

                # Positive links
                pos_links = snapshot.edge_index.t()
                pos_labels = torch.ones(pos_links.size(0))

                # Negative links
                neg_links = negative_sampling(snapshot.edge_index, embeddings.size(0), pos_links.size(0))
                neg_labels = torch.zeros(neg_links.size(0))

                # Concatenate positive and negative links
                links = torch.cat([pos_links, neg_links], dim=0)
                labels = torch.cat([pos_labels, neg_labels])
                
                scores = (embeddings[links[:, 0]] * embeddings[links[:, 1]]).sum(dim=1)
                
                loss = criterion(scores, labels)
                loss.backward()
                optimizer.step()

                total_loss += loss.item()
            
            avg_loss = total_loss / len(train_subset)
            print(f"Epoch {epoch+1}, Loss: {avg_loss}")
            total_loss = 0

            # Validation
            model.eval()
            val_loss = 0
            with torch.no_grad():
                for snapshot in val_subset:
                    embeddings = model(snapshot.x, snapshot.edge_index).squeeze()
                    pos_links = snapshot.edge_index.t()
                    pos_labels = torch.ones(pos_links.size(0))
                    neg_links = negative_sampling(snapshot.edge_index, embeddings.size(0), pos_links.size(0))
                    neg_labels = torch.zeros(neg_links.size(0))
                    links = torch.cat([pos_links, neg_links], dim=0)
                    labels = torch.cat([pos_labels, neg_labels])
                    scores = (embeddings[links[:, 0]] * embeddings[links[:, 1]]).sum(dim=1)
                    val_loss += criterion(scores, labels).item()
            
            avg_val_loss = val_loss / len(val_subset)
            print(f"Epoch {epoch+1}, Validation Loss: {avg_val_loss}")
            
            # Early stopping
            if avg_val_loss < best_val_loss:
                best_val_loss = avg_val_loss
                patience_counter = 0
                best_model_state = model.state_dict()
            else:
                patience_counter += 1
                if patience_counter >= patience:
                    print("Early stopping triggered")
                    break

        # Load the best model
        model.load_state_dict(best_model_state)

        # Evaluate on test dataset
        results = []
        for snapshot in test_dataset:
            pos_links = snapshot.edge_index.t()
            pos_labels = torch.ones(pos_links.size(0)).cpu().numpy()
            neg_links = negative_sampling(snapshot.edge_index, embeddings.size(0), pos_links.size(0))
            neg_labels = torch.zeros(neg_links.size(0)).cpu().numpy()
            true_edges = torch.cat([pos_links, neg_links], dim=0)
            labels = np.concatenate([pos_labels, neg_labels])
            result = eval_link_predictor_with_plots(model, snapshot, true_edges, labels)
            results.append(result)

        avg_results = {
            'AUC-ROC': np.mean([res['AUC-ROC'] for res in results]),
            'AUC-PR': np.mean([res['AUC-PR'] for res in results]),
            'F1': np.mean([res['F1'] for res in results])
        }
        auc_rocs.append(avg_results['AUC-ROC'])
        auc_prs.append(avg_results['AUC-PR'])
        f1_scores.append(avg_results['F1'])
        all_results.append(avg_results)

    return {
        'AUC-ROC': (np.mean(auc_rocs), np.std(auc_rocs)),
        'AUC-PR': (np.mean(auc_prs), np.std(auc_prs)),
        'F1': (np.mean(f1_scores), np.std(f1_scores))
    }, all_results

In [ ]:
# Running Monte Carlo experiments
num_experiments = 1  # You can increase this number for more robust results
monte_carlo_results, all_results = monte_carlo_experiment(data_list, specific_train_files, num_experiments=num_experiments)

# Display the results
print("Monte Carlo Results:")
print(f"AUC-ROC: Mean = {monte_carlo_results['AUC-ROC'][0]:.3f}, Std = {monte_carlo_results['AUC-ROC'][1]:.3f}")
print(f"AUC-PR: Mean = {monte_carlo_results['AUC-PR'][0]:.3f}, Std = {monte_carlo_results['AUC-PR'][1]:.3f}")
print(f"F1: Mean = {monte_carlo_results['F1'][0]:.3f}, Std = {monte_carlo_results['F1'][1]:.3f}")

Epoch 104, Loss: 85.90230821308337
Epoch 104, Validation Loss: 21.60982551574707
Epoch 105, Loss: 85.08506694592927
Epoch 105, Validation Loss: 15.68709774017334
Epoch 106, Loss: 81.75674358167146
Epoch 106, Validation Loss: 19.669495010375975
Epoch 107, Loss: 81.0016636095549
Epoch 107, Validation Loss: 11.471026420593262
Epoch 108, Loss: 79.04774695948551
Epoch 108, Validation Loss: 15.81252384185791
Epoch 109, Loss: 78.11093982897307
Epoch 109, Validation Loss: 15.799080848693848
Epoch 110, Loss: 76.2055933099044
Epoch 110, Validation Loss: 17.022938346862794
Epoch 111, Loss: 73.6551852979158
Epoch 111, Validation Loss: 14.111651039123535
Epoch 112, Loss: 73.34890064440276
Epoch 112, Validation Loss: 16.357756042480467
Epoch 113, Loss: 71.53133402372661
Epoch 113, Validation Loss: 16.67223491668701
Epoch 114, Loss: 69.65285451788651
Epoch 114, Validation Loss: 12.604134368896485
Epoch 115, Loss: 67.4015921542519
Epoch 115, Validation Loss: 17.252034759521486
Epoch 116, Loss: 67.1955

In [ ]:
predicted_adj_matrices_ = {}

for snapshot in train_dataset:
    model.eval()
    with torch.no_grad():
        # Get node embeddings
        embeddings = model(snapshot.x, snapshot.edge_index)

        # Initialize an empty adjacency matrix
        num_nodes = embeddings.shape[0]
        directed_adj_matrix = torch.zeros((num_nodes, num_nodes))

        # Collect all link scores
        all_link_scores = []

        # Compute link scores for potential directed edges
        for i in range(num_nodes):
            for j in range(num_nodes):
                if i != j:  # Exclude self-loops in this calculation
                    link_score_ij = torch.sigmoid(torch.dot(embeddings[i], embeddings[j]))
                    all_link_scores.append(link_score_ij.item())  # Collect link score

        # Calculate the threshold
        threshold = torch.quantile(torch.tensor(all_link_scores), 0.75).item()

        # Apply the threshold
        for i in range(num_nodes):
            for j in range(num_nodes):
                if i != j:
                    link_score_ij = torch.sigmoid(torch.dot(embeddings[i], embeddings[j]))
                    directed_adj_matrix[i, j] = 1 if link_score_ij > threshold else 0

        # Post-processing to enforce directionality and add self-loops
        for i in range(num_nodes):
            for j in range(num_nodes):
                if i != j:
                    if directed_adj_matrix[i, j] == 1 and directed_adj_matrix[j, i] == 1:
                        # Keep only the edge with the higher link score
                        score_ij = torch.sigmoid(torch.dot(embeddings[i], embeddings[j])).item()
                        score_ji = torch.sigmoid(torch.dot(embeddings[j], embeddings[i])).item()
                        if score_ij > score_ji:
                            directed_adj_matrix[j, i] = 0
                        else:
                            directed_adj_matrix[i, j] = 0
                else:
                    # Ensure diagonal values are set to 1 for self-loops
                    directed_adj_matrix[i, i] = 1

        # Store the predicted adjacency matrix
        predicted_adj_matrices_[snapshot.file_name] = directed_adj_matrix.cpu().numpy()

In [ ]:
# utilisation de n'importe matrice pour extraire les noms de colonnes
df = adj_matrices[0]

# Extract node names from the column headers
node_names = df.columns.tolist()

In [ ]:
# To access a specific matrix by filename:
specific_filename = 'rectified_matrix_2016-11-04.xlsx'

# Now you have the node names, you can label your adjacency matrix
specific_predicted_matrix = predicted_adj_matrices_.get(specific_filename, None)

if specific_predicted_matrix is not None:
    # Convert the numpy array to a pandas DataFrame with node names as labels
    specific_predicted_df = pd.DataFrame(specific_predicted_matrix, 
                                         index=node_names, 
                                         columns=node_names)
else:
    print("Matrix not found for the specified file name.")

specific_predicted_df

In [ ]:
specific_predicted_df.to_excel("predicted_matrix_2016-11-04.xlsx")

In [ ]:
# Specific columns you want to display
specific_columns = ['MSFT', 'GOOGL', 'DIS', 'AMZN', 'SBUX', 'PFE', 'JNJ', 'ABBV', 'JPM', 'BAC', 'GS']

desired_data = adj_matrices[1].loc[specific_columns, specific_columns]
predicted_data = specific_predicted_df.loc[specific_columns, specific_columns]

# Print the data
print(desired_data)
print(predicted_data)